# Data Description

### Main parts of the dataset

- **User tables**
  - `students.csv`, `teachers.csv`
  - user identifiers, group membership, account creation time

- **Platform activity**
  - `pageviews.csv`
  - `events/*.csv`
  - navigation and fine-grained interaction logs (clicks, scroll, question views, media events, chatbot open/close)

- **Chatbot usage**
  - `gymitrainer.csv`
  - `gymitrainer_feedback.csv`
  - conversation threads with the AI chatbot and user feedback on those interactions

- **Performance / outcome tables**
  - `quiz_results.csv`
  - `math_results.csv`
  - `text_results.csv`
  - `essay_results.csv`
  - detailed student results across different learning tasks

- **Supporting / mapping tables**
  - `course_ids.csv`
  - `math_questions.csv`
  - `quiz_questions.csv`
  - `text_questions.csv`
  - `essay_feedback.csv`

### Most important identifiers

The main linking variable across the dataset is usually:

- `user_id`

Other useful keys include:

- `result_id` for essay submissions
- `thread_id` for chatbot feedback
- `question_id` for question-level results
- `url` / `post_id` for linking pages to course content

### Practical note

The dataset mixes several time formats:
- Unix timestamps in **seconds**
- client-side event timestamps in **milliseconds**
- datetime strings in `pageviews.csv`

## First Research Question

**What are the effects of students’ chatbot engagement and on-platform focus behavior on their academic performance?**

More specifically, we will study whether behavioral indicators extracted from `gymitrainer.csv`, `gymitrainer_feedback.csv`, and the event logs (especially `events/g` and other relevant `events/*.csv` files) are associated with student performance in quizzes and essays, as measured from `quiz_results.csv` and `essay_results.csv`.

Our goal is to model and compare how different forms of interaction with the platform — such as chatbot usage, interaction intensity, and focus-related behavioral traces — relate to learning outcomes.

In [24]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [25]:
# Directory
print("Python working directory:", os.getcwd())

data_dir = './data'
assert os.path.isdir(data_dir), f"File not found: {data_dir}" # check files exist

Python working directory: /Users/louistschanz/Documents/EPFL/MA4/MLBD_2026


## Loading the Data by type

In [26]:
# 1. User Tables
students = pd.read_csv('{}/students.csv'.format(data_dir))
teachers = pd.read_csv('{}/teachers.csv'.format(data_dir))

In [5]:
# 2. Platform activity tables
pageviews = pd.read_csv('{}/pageviews.csv'.format(data_dir))
## events will do later

In [6]:
# 3. Chatbots specific tables
gymitrainer_feedback = pd.read_csv('{}/gymitrainer_feedback.csv'.format(data_dir))
gymitrainer = pd.read_csv('{}/gymitrainer.csv'.format(data_dir))

In [59]:
# 4. Learning outcomes tables / Perforamance tables
math_results = pd.read_csv('{}/math_results.csv'.format(data_dir))
quiz_results = pd.read_csv('{}/quiz_results.csv'.format(data_dir))
text_results = pd.read_csv('{}/text_results.csv'.format(data_dir))
essay_results = pd.read_csv('{}/essay_results.csv'.format(data_dir))

In [ ]:
# 5. Supporting / Mapping / metadata tables // help connect the other tables
course_ids = pd.read_csv('{}/course_ids.csv'.format(data_dir))
math_questions = pd.read_csv('{}/math_questions.csv'.format(data_dir))
quiz_questions = pd.read_csv('{}/quiz_questions.csv'.format(data_dir))
text_questions = pd.read_csv('{}/text_questions.csv'.format(data_dir))
essay_feedback = pd.read_csv('{}/essay_feedback.csv'.format(data_dir))
comments = pd.read_csv('{}/comments.csv'.format(data_dir))

## Creating the Tables

In [63]:
print("Number of distincts users in math submissions :", math_results['user_id'].nunique(),"users")
print("Number of distincts users in quiz submissions :", quiz_results['user_id'].nunique(),"users")
print("Number of distincts users in text submissions :", text_results['user_id'].nunique(),"users")
print("Number of distincts users in essay submissions :", essay_results['user_id'].nunique(),"users")

Number of distincts users in math submissions : 227 users
Number of distincts users in quiz submissions : 1606 users
Number of distincts users in text submissions : 647 users
Number of distincts users in essay submissions : 1402 users


In [69]:
math_results.head()

,Unnamed: 0,session_id,question_part,user_id,question_id,points,max_points,answer,timestamp
0,0,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63e3,0.0,4.0,NaN,1738158044
1,1,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63e1,0.0,2.0,NaN,1738158044
2,2,679a2fdce127578405afeef5,1,490,66a5eb558948dda8b2fd63e1,0.0,2.0,NaN,1738158044
3,3,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63df,4.0,4.0,21,1738158044
4,4,679a2fdce127578405afeef5,0,490,66a5eb558948dda8b2fd63e0,0.0,4.0,NaN,1738158044


In [ ]:
perf_by_users = pd.DataFrame({
    'user_id': pd.unique(
        pd.concat([math_results['user_id'], text_results['user_id'],quiz_results['user_id'],essay_results['user_id']], ignore_index=True)
    )
})

perf_by_users = perf_by_users.sort_values('user_id').reset_index(drop=True)
perf_by_users.head()

In [ ]:
# Compute absolute points column
avg_point_math_absolute = quiz_results.groupby('user_id')['points'].mean().reset_index(name='average_point_math_absolute') # reset index so user_id is not an index

# Compute relative points column
# Fix inconsistent rows (Hint 2)
quiz_results.loc[quiz_results['points'] > quiz_results['max_points'], 'max_points'] = \
    quiz_results.loc[quiz_results['points'] > quiz_results['max_points'], 'points']

# Fix when max points = 0
quiz_valid = quiz_results[quiz_results['max_points'] > 0].copy() # remove questions if max_point=0, it means they don't count
quiz_valid['points_relative'] = quiz_valid['points'] / quiz_valid['max_points'] # compute relative score
average_point_math_relative = (quiz_valid.groupby('user_id')['points_relative'].mean().reset_index(name='average_point_math_relative')) # average per user

# Merge absolute math points
perf_by_users = perf_by_users.merge(avg_point_math_absolute, on='user_id', how='left')

# Merge relative math points
perf_by_users = perf_by_users.merge(average_point_math_relative, on='user_id', how='left')

In [77]:
perf_by_users.sample(10)

,user_id,average_point_math_absolute,average_point_math_relative
1585,6228,NaN,NaN
1756,6460,0.770115,0.772201
865,1485,0.456140,0.464286
1252,1974,0.669683,0.675799
1851,6586,0.719512,0.719512
437,937,NaN,NaN
573,1121,0.742424,0.753846
1571,6190,0.671053,0.671053
1709,6395,1.000000,1.000000
1300,2038,0.716049,0.725000


In [76]:
# check if user_id is in math_results
user_id_to_check = 491  # replace with the user id you want to inspect

if user_id_to_check in math_results['user_id'].values:
    print(f"{user_id_to_check} is in math_results")
    display(math_results[math_results['user_id'] == user_id_to_check].head())
else:
    print(f"{user_id_to_check} is not in math_results")

491 is not in math_results
